# 44 · Label-free sensor preparation audit

Build completed 30-second epochs from raw heart-rate and motion observations for development participants. Compare seven sensor channels and coverage measurements with saved research preparation. The audit separately measures changes in epoch eligibility and history length.

The configured session origin is zero seconds. The earlier of the two final sensor timestamps bounds the schedule, so the unfinished tail is excluded. The final-test participants remain reserved, and the audit never opens raw sleep-stage files.

A numerical pass establishes parity on the compared values. Online model context equivalence and clinical accuracy require separate evaluation.


## Locate the checkout and set run options

**What:** Resolve the project root and set the setup, audit, and participant timeline options.

**Why:** With both run flags off, the notebook reads metadata and displays a verified snapshot when one exists.


In [ ]:
import logging
import os
import shutil
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import display

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
logger = logging.getLogger("label_free_sensor_audit_notebook")
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "configs/label-free-sensor-audit.yaml").is_file()),
    Path.cwd() / "sleep-classifier",
)
ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", str(ROOT))).expanduser().resolve()

REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "feature/label-free-sensor-audit"
RUN_SETUP = False
RUN_AUDIT = False
TIMELINE_PARTICIPANT = None  # Set to a development participant ID as a string

logger.info("Repository=%s setup=%s audit=%s", ROOT, RUN_SETUP, RUN_AUDIT)


## Set up the environment when requested

**What:** Clone the feature branch if the checkout is missing and install the locked baseline, notebook, and development groups.

**Why:** The audit uses the same environment on a new checkout and preserves existing optional groups.


In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv download failed")
        !sh "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --inexact --group baseline --group notebook --group dev
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")
else:
    logger.info("Environment setup skipped")


## Check the audit plan

**What:** Inspect the configured clock, comparison tolerance, and participant membership.

**Why:** The plan records the allowed scope before any observations are read.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from sleep_classifier.sensor_preparation_audit import LabelFreeSensorAudit

audit = LabelFreeSensorAudit(ROOT / "configs/label-free-sensor-audit.yaml")
plan = audit.plan()
logger.info("Audit config=%s output=%s", audit.config_path, audit.output_dir)
display(pd.DataFrame([
    {
        "development_participants": len(plan["development_participants"]),
        "reserved_test_participants": len(plan["reserved_test_participants"]),
        "session_start_seconds": plan["session_start_seconds"],
        "epoch_seconds": plan["epoch_seconds"],
        "end_policy": plan["end_policy"],
        "context_epochs": plan["context_epochs"],
        "absolute_tolerance": plan["absolute_tolerance"],
    }
]))
display(pd.DataFrame([{
    key: plan[key] for key in (
        "model_loaded", "training_performed", "test_observations_read", "raw_labels_read"
    )
}]))


## Run the audit only when requested

**What:** Create a new audit snapshot when the run flag is enabled and the output directory is absent.

**Why:** The audit reads allowed raw development sensor files and publishes an immutable snapshot. Existing snapshots are verified in the next cell.


In [ ]:
if RUN_AUDIT and not audit.output_dir.exists():
    logger.info("Run audit config=%s output=%s", audit.config_path, audit.output_dir)
    %cd "{ROOT}"
    !uv run --locked --inexact --group baseline --group notebook python -u \
        -m sleep_classifier.sensor_preparation_audit --config "{audit.config_path}" --run
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Label-free sensor audit failed")
elif audit.output_dir.exists():
    logger.info("Use the existing audit snapshot at %s", audit.output_dir)
else:
    logger.info("Audit skipped; RUN_AUDIT is disabled")


## Load and verify the saved snapshot

**What:** Verify the configuration and saved report hashes before reading the diagnostic rows and summary.

**Why:** The charts and tables use verified results without rerunning the audit.


In [ ]:
rows = None
summary = None
if audit.output_dir.exists():
    rows, summary = audit.load_report()
    logger.info("Verified audit snapshot at %s", audit.output_dir)
    display(pd.DataFrame([{
        "numerical_status": summary["status"],
        "development_participants": len(summary["participants"]),
        "snapshot_rows": len(rows),
    }]))
else:
    logger.info("Audit snapshot pending; enable RUN_AUDIT to generate it")


## Inspect epoch scope and context changes

**What:** Count clock-only, shared, and reference-only epochs. Plot label-only exclusions and history-length changes by participant.

**Why:** Eligibility and history can change even when the shared numerical summaries agree.

Context-cause counts can overlap. They describe retained history; they do not measure prediction changes.


In [ ]:
if summary is not None:
    scopes = {
        "left_only": "Clock only",
        "both": "Shared",
        "right_only": "Prepared reference only",
    }
    scope_counts = rows.epoch_scope.value_counts().reindex(scopes, fill_value=0)
    display(scope_counts.rename(index=scopes).rename("epochs").to_frame())

    participants = pd.DataFrame(summary["participants"]).set_index("participant_id")
    measures = {
        "label_only_exclusions": "Label-only exclusions",
        "context_changed_epochs": "History length changed",
    }
    ax = participants[list(measures)].rename(columns=measures).plot.bar(
        figsize=(13, 4), color=["#485c86", "#bd632f"], width=0.8
    )
    ax.set(title="Eligibility and history changes", xlabel="Participant ID", ylabel="Epochs")
    ax.figure.tight_layout()
    plt.show()

if rows is not None:
    context_causes = audit.context_causes(rows)
    display(context_causes.set_index("participant_id"))


## Inspect numerical parity

**What:** Show mismatch counts and maximum absolute differences on shared epochs. Show the separate historical fixed-lag motion comparison.

**Why:** These checks quantify feature and coverage agreement independently of label exclusions and history changes.


In [ ]:
if summary is not None:
    feature_checks = pd.DataFrame(participants.feature_mismatches.tolist())
    feature_differences = pd.DataFrame(participants.maximum_feature_differences.tolist())
    display(pd.DataFrame({
        "mismatches": feature_checks.sum(),
        "maximum_absolute_difference": feature_differences.max(),
    }).rename_axis("feature_or_coverage_measure"))
    display(pd.DataFrame([{
        "eligibility_rule_mismatches": participants.eligibility_rule_mismatches.sum(),
        **summary["motion_parity"],
    }], index=["Eligibility and fixed-lag motion checks"]))


## View one participant timeline

**What:** If a participant ID is set, plot eligibility and history length along that participant's epoch timeline.

**Why:** The timeline places changes in consecutive context beside the eligibility decisions that precede them.


In [ ]:
if rows is not None and TIMELINE_PARTICIPANT is not None:
    selected = rows.loc[rows.participant_id.eq(TIMELINE_PARTICIPANT)].sort_values("time")
    if selected.empty:
        raise ValueError("Choose a development participant from the snapshot")

    fig, axes = plt.subplots(2, 1, sharex=True, figsize=(13, 5))
    for ax, column in zip(axes, ("eligible", "history_epochs"), strict=True):
        ax.step(
            selected.time, pd.to_numeric(selected[column]),
            where="post", color="#485c86", label="Sensor clock",
        )
        ax.step(
            selected.time, pd.to_numeric(selected[f"{column}_reference"]),
            where="post", color="#bd632f", linestyle="--", label="Prepared reference",
        )
    axes[0].set(title=f"Participant {TIMELINE_PARTICIPANT}", ylabel="Eligibility",
                yticks=[0, 1], yticklabels=["Ineligible", "Eligible"])
    axes[0].legend()
    axes[1].set(xlabel="Epoch start (seconds)", ylabel="History (epochs)")
    fig.tight_layout()
    plt.show()


## Observed development audit · 2026-11-24

The numerical audit passed on shared epochs without loading a model or running training.
All six base features and six coverage measurements matched at absolute tolerance 1e-8.
Fixed-lag motion matched exactly on 22,705 saved epochs. Three saved motion epochs fell outside the conservative clock boundary.

The clock produced 24,738 epochs across 26 development participants, including 915 outside the saved label schedule.
Of those clock-only epochs, 589 met the sensor coverage rule. Another 380 shared epochs had coverage but no scored target.
Keeping covered unscored epochs changes the bounded history of 566 shared eligible epochs across fifteen participants.
The cached context diagnostics attribute all 566 to unscored epochs in their retained history.

These are preparation results. No model predictions or accuracy values were recomputed.
The session origin is declared as zero. The earlier final sensor timestamp closes epochs conservatively; unfinished tails remain excluded.
This batch audit does not implement live arrival handling, clock synchronization, fitted imputation, or a streaming service.
The final test remains reserved. See [the audit guide](../docs/label-free-sensor-audit.md) for scope and next steps.


## Export a report-only HTML copy

**What:** Execute report cells in a fresh kernel and save an HTML copy.

**Why:** The report skips setup, audit execution, and recursive export. The source notebook keeps empty outputs.


In [ ]:
%cd "{ROOT}"
!uv run --locked --inexact --group baseline --group notebook python scripts/execute_notebook.py \
    --notebook notebooks/44_label_free_sensor_audit.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Label-free sensor audit notebook export failed")
